In [0]:
# Retail Sales Lakehouse
# End-to-End Data Validation

validation_tables = {
    "Bronze Customers": "workspace.default.bronze_customers",
    "Silver Customers": "workspace.default.silver_customers",

    "Bronze Products": "workspace.default.bronze_products",
    "Silver Products": "workspace.default.silver_products",

    "Bronze Orders": "workspace.default.bronze_orders",
    "Silver Orders": "workspace.default.silver_orders",

    "Bronze Order Items": "workspace.default.bronze_order_items",
    "Silver Order Items": "workspace.default.silver_order_items"
}

for name, table in validation_tables.items():
    count = spark.table(table).count()
    print(f"{name}: {count} rows")

In [0]:
checks = []

# 1. Customer row count reconciliation
bronze_customers_count = spark.table("workspace.default.bronze_customers").count()
silver_customers_count = spark.table("workspace.default.silver_customers").count()

checks.append(
    ("Customer row count", bronze_customers_count == silver_customers_count)
)

# 2. Product row count reconciliation
bronze_products_count = spark.table("workspace.default.bronze_products").count()
silver_products_count = spark.table("workspace.default.silver_products").count()

checks.append(
    ("Product row count", bronze_products_count == silver_products_count)
)

# 3. Orders row count reconciliation
bronze_orders_count = spark.table("workspace.default.bronze_orders").count()
silver_orders_count = spark.table("workspace.default.silver_orders").count()

checks.append(
    ("Orders row count", bronze_orders_count == silver_orders_count)
)

# 4. Order items row count reconciliation
bronze_order_items_count = spark.table("workspace.default.bronze_order_items").count()
silver_order_items_count = spark.table("workspace.default.silver_order_items").count()

checks.append(
    ("Order items row count", bronze_order_items_count == silver_order_items_count)
)

for check_name, result in checks:
    status = "PASS" if result else "FAIL"
    print(f"{check_name}: {status}")

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

gold_fact = spark.table("workspace.default.gold_fact_sales")

revenue_check = (
    gold_fact
    .withColumn(
        "expected_revenue",
        col("quantity") * col("unit_price")
    )
    .filter(
        col("revenue") != col("expected_revenue")
    )
)

display(revenue_check)

In [0]:
gold_fact = spark.table("workspace.default.gold_fact_sales")

# Null validation
null_gold = gold_fact.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("product_id").isNull() |
    col("quantity").isNull() |
    col("unit_price").isNull() |
    col("revenue").isNull()
)

# Duplicate validation
duplicate_gold = (
    gold_fact
    .groupBy("order_item_id")
    .count()
    .filter(col("count") > 1)
)

print("Null rows:", null_gold.count())
print("Duplicate order_item_id rows:", duplicate_gold.count())

In [0]:
summary_checks = {
    "Bronze tables available": 4,
    "Silver tables available": 4,
    "Gold tables available": 5,
    "Customer row reconciliation": "PASS",
    "Product row reconciliation": "PASS",
    "Orders row reconciliation": "PASS",
    "Order items row reconciliation": "PASS",
    "Gold revenue validation": "PASS",
    "Gold null validation": "PASS",
    "Gold duplicate validation": "PASS"
}

for check, result in summary_checks.items():
    print(f"{check}: {result}")